In [1]:
# /// script
# dependencies = [
#   "microcal[ndv-jup] @ git+https://github.com/fdrgsp/microcal",
# ]
# ///

In [2]:
from microcal import ChromaticShiftCorrector
from rich import print
import ndv
from microcal import generate_beads_image

In [3]:
import logging
logging.basicConfig(level=logging.INFO, format="%(name)s | %(levelname)s | %(message)s")

In [4]:
# generate a 2-channel synthetic beads image
beads_img, _ = generate_beads_image(
    n_channels=2,
    shape=(512, 512),
    n_beads=50,
    bead_sigma=2,
    bead_intensity=60.0,
    bit_depth=16,
    offset=100,
    shifts=[(0, 0), (1.5, -2.5)],
    rotations=[0, 5],
    scales=[(1, 1), (1.05, 0.95)],
    snr=8,
    seed=42
)

# visualize the synthetic beads image with ndv
ndv.imshow(
    beads_img,
    channel_mode="composite",
    luts={0: {"cmap": "green"}, 1: {"cmap": "magenta"}},
)

RFBOutputContext()

<IPython.core.display.Javascript object>

RFBOutputContext()

RFBOutputContext()

<IPython.core.display.Javascript object>

In [5]:
# initialize the chromatic shift corrector with appropriate parameters
sc = ChromaticShiftCorrector(
    reference_channel=0,
    smooth_sigma=3,
    min_distance=2,
    threshold_rel=0.5,
    match_max_distance=50,
    min_pairs=2,
    subpixel_refine=True,
    refine_radius=2,
)

In [6]:
# measure the chromatic shift on the synthetic beads image
results = sc.measure(beads_img)

microcal._chromatic_shift_corrector | INFO | ch0 (reference): 50 beads detected
microcal._chromatic_shift_corrector | INFO | ch1: 48 beads detected | coarse shift (row, col) = [10.6   8.97]
microcal._chromatic_shift_corrector | INFO | ch1: 41 bead pairs matched (max_distance=50.0px)
microcal._chromatic_shift_corrector | INFO | ch1: fit RMS = 0.322px  transform =
[[  1.04896762   0.08309219 -31.29314132]
 [ -0.09190656   0.94869429  34.92256455]
 [  0.           0.           1.        ]]


In [7]:
# visualize the detected beads in the first (reference) channel
ch1_det = results.detection_image[:2, :, :]
# in this image, 0 is the reference channel, and 1 is the beads mask
ndv.imshow(
    ch1_det,
    channel_mode="composite",
    luts={0: {"cmap": "green"}, 1: {"cmap": "gray"}},
)

RFBOutputContext()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# visualize the detected beads in the second channel
ch2_det = results.detection_image[2:4, :, :]
# in this image, 2 is the second channel, and 3 is the beads mask
ndv.imshow(
    ch2_det,
    channel_mode="composite",
    luts={2: {"cmap": "magenta"}, 3: {"cmap": "gray"}},
)

RFBOutputContext()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
# visualize the matched bead pairs between the two channels
ndv.imshow(results.pairs_image.astype("uint16"), default_lut={"cmap": "glasbey"})

RFBOutputContext()

<IPython.core.display.Javascript object>

In [13]:
# print the validation results of the measured chromatic shift
val = sc.validate()

microcal._chromatic_shift_corrector | INFO | 
Validation report  (reference = channel 0)
 Channel   N pairs   Mean err (px)   Median (px)    Max (px)   Std (px)
----------------------------------------------------------------------
       1        48           0.093         0.070       0.272      0.060


In [14]:
# apply the measured chromatic shift correction to another image (in this case,
# the same synthetic beads image)
image_corr = sc.apply(image_or_stack=beads_img, result=results, crop=True)
ndv.imshow(
    image_corr,
    channel_mode="composite",
    luts={0: {"cmap": "green"}, 1: {"cmap": "magenta"}},
)

RFBOutputContext()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>